<img src="../../../assets/images/logos/ucu_logo_clean.svg" alt="UCU Logo" width="200" style="float: right; margin: 0 0 10px 10px;"/>

### Matemáticas para Aprendizaje Automático - 2026

--------
## Laboratorio 1.3: Autovalores, Autovectores y Métodos Iterativos

#### Objetivos

- Calcular descomposiciones espectrales con `numpy.linalg` y decidir entre `eig` y `eigh` según la estructura de la matriz, justificando la diferencia en precisión y en costo.
- Implementar el método de la potencia y sus variantes (potencia inversa y potencia inversa con desplazamiento), y relacionar su velocidad de convergencia con la separación entre autovalores.
- Diagnosticar los casos en que los métodos iterativos fallan: autovalores de igual módulo, autovalores repetidos y matrices no diagonalizables.
- Obtener los $k$ autovectores dominantes por deflación y usarlos para calcular las direcciones principales de un conjunto de datos.

## Introducción

Un autovector de $A \in \mathbb{R}^{n \times n}$ es una dirección que la matriz no cambia: sólo la estira o la contrae por un factor $\lambda$, el autovalor asociado. Esa idea, que en un curso de álgebra lineal aparece como una definición más, es el motor de buena parte del aprendizaje automático. Las direcciones de máxima varianza de un conjunto de datos son los autovectores de su matriz de covarianza (PCA); la curvatura de una función de pérdida en un mínimo está descrita por los autovalores de su Hessiano, que determinan qué tan rápido converge el descenso por gradiente; el ranking de páginas de PageRank es el autovector dominante de una matriz de transición; y la estabilidad de un sistema dinámico depende del módulo de sus autovalores.

En la práctica, los problemas de interés involucran matrices grandes, y casi nunca se necesita el espectro completo: alcanza con unos pocos autovectores, los asociados a los autovalores de mayor módulo. Calcular las $n$ raíces del polinomio característico no sólo es carísimo, sino numéricamente inestable (pequeñas perturbaciones en los coeficientes producen cambios enormes en las raíces). Los algoritmos que se usan de verdad son iterativos: parten de un vector cualquiera y lo mejoran aplicando la matriz una y otra vez, con un costo de $O(n^2)$ por iteración en lugar de $O(n^3)$ por la descomposición completa.

En este laboratorio empezamos por las rutinas de NumPy (`eig` y `eigh`) y su verificación, seguimos implementando desde cero el método de la potencia y sus variantes, estudiamos cuándo y por qué fallan, y cerramos usándolos para calcular las direcciones principales de un conjunto de datos, la puerta de entrada al Análisis de Componentes Principales que se retoma en el Laboratorio 1.4.

In [ ]:
# Librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
import timeit


def residuo_autopar(A, lam, v):
    """
    Mide qué tan bueno es el autopar (lam, v) para la matriz A:

        r(lam, v) = || A v - lam v || / || v ||

    Esta función viene dada y se usa en todo el laboratorio como test numérico:
    un residuo del orden de la precisión de máquina indica que v es, numéricamente,
    un autovector de A con autovalor lam.
    """
    v = np.asarray(v).ravel()
    return np.linalg.norm(A @ v - lam * v) / np.linalg.norm(v)


## Parte 1: Descomposición Espectral con NumPy

Un vector no nulo $v \in \mathbb{C}^n$ es **autovector** de $A \in \mathbb{R}^{n\times n}$ con **autovalor** $\lambda \in \mathbb{C}$ si

$$A v = \lambda v, \qquad v \neq 0.$$

Equivalentemente, $(A - \lambda I)v = 0$ con $v \neq 0$, de modo que $\lambda$ es autovalor si y sólo si $\det(A - \lambda I) = 0$. Ese determinante es el **polinomio característico** de $A$, de grado $n$. Sirve para calcular a mano el espectro de una matriz $2 \times 2$, pero es una pésima herramienta numérica: los coeficientes del polinomio son extremadamente sensibles a perturbaciones de la matriz, y sus raíces lo son aún más. Ningún software serio lo usa.

Si $A$ tiene $n$ autovectores linealmente independientes es **diagonalizable**: agrupándolos como columnas de $P$ y los autovalores en $D = \operatorname{diag}(\lambda_1,\dots,\lambda_n)$,

$$A = P D P^{-1}.$$

No toda matriz lo es. El ejemplo mínimo es $\begin{bmatrix} 2 & 1 \\ 0 & 2\end{bmatrix}$, que tiene $\lambda = 2$ con multiplicidad algebraica $2$ pero un único autovector independiente. Además, una matriz real puede tener autovalores complejos (una rotación del plano no deja ninguna dirección fija).

**El caso simétrico.** Si $A = A^\top$, el **teorema espectral** garantiza mucho más: todos los autovalores son reales y existe una base **ortonormal** de autovectores, es decir

$$A = Q \Lambda Q^\top, \qquad Q^\top Q = I .$$

Ésta es la situación de las matrices de covarianza, los Hessianos y los Gramianos $X^\top X$, que además son semidefinidas positivas ($\lambda_i \ge 0$).

**Las rutinas de NumPy.** `numpy.linalg.eig` sirve para matrices generales: devuelve `(valores, vectores)` con los autovectores como **columnas** de la matriz devuelta, normalizados a norma $1$, en orden arbitrario y con tipo complejo si hace falta. `numpy.linalg.eigh` está pensada para matrices simétricas (o hermíticas): explota la estructura para trabajar a aproximadamente la mitad del costo, devuelve autovalores reales **ordenados de menor a mayor** y una base ortonormal garantizada. Las variantes `eigvals` y `eigvalsh` calculan sólo los autovalores, más rápido, cuando los autovectores no hacen falta.

Hay una trampa importante: `eigh` **lee sólo un triángulo** de la matriz (por defecto el inferior, `UPLO='L'`) y asume que el otro es su reflejo. Si se la aplica a una matriz que no es simétrica, no da error: devuelve, silenciosamente, la descomposición de *otra* matriz. Verificar es imprescindible, y para eso está la función `residuo_autopar` de la celda de imports.

### Ejercicio L1.3.1: Descomposición Espectral y Verificación

Las rutinas de NumPy devuelven los autovalores en un orden que no controlamos. Como casi siempre nos interesan los autovalores **dominantes** (los de mayor módulo), conviene envolverlas en una función que ordene el resultado.

**a)** Implementá `autopares(A, simetrica=False)`, que calcula la descomposición espectral de `A` con `numpy.linalg.eigh` si `simetrica` es `True` y con `numpy.linalg.eig` en caso contrario, y devuelve la tupla `(valores, vectores)` reordenada por $|\lambda|$ **decreciente**. Los autovectores son las columnas de `vectores` y deben quedar en el mismo orden que los autovalores.

**b)** Aplicá la función a

$$A = \begin{bmatrix} 1 & 2 \\ 3 & 1 \end{bmatrix}$$

y calculá el residuo `residuo_autopar` de cada autopar obtenido.

**Nota:** `np.argsort` ordena de menor a mayor; para invertir el orden se usa `[::-1]`. Al reordenar los autovectores hay que permutar **columnas**: `vectores[:, orden]`. El resultado analítico es $\lambda = 1 \pm \sqrt{6}$, que sale de $\det(A - \lambda I) = (1-\lambda)^2 - 6 = 0$.

In [ ]:
def autopares(A, simetrica=False):
    """
    Calcula la descomposición espectral de A, ordenada por |lambda| decreciente.

    Args:
        A: matriz cuadrada (n x n)
        simetrica: si es True usa numpy.linalg.eigh; si es False, numpy.linalg.eig

    Returns:
        valores: autovalores ordenados por módulo decreciente (array de largo n)
        vectores: matriz cuyas COLUMNAS son los autovectores, en el mismo orden (n x n)
    """
    valores, vectores = ...  # COMPLETAR: elegir eigh o eig según el argumento simetrica
    orden = ...              # COMPLETAR: índices que ordenan |valores| de mayor a menor
    return valores[orden], vectores[:, orden]


# b) Aplicación a la matriz del enunciado
A = np.array([[1.0, 2.0],
              [3.0, 1.0]])

vals, vecs = ...  # COMPLETAR: autopares de A (no es simétrica)
residuos = ...    # COMPLETAR: array con el residuo de cada autopar (vals[i], vecs[:, i])

# ── Verificación ─────────────────────────────────────────────────────────────
esperado = np.array([1 + np.sqrt(6), 1 - np.sqrt(6)])   # raíces de (1-λ)^2 = 6

assert np.allclose(vals, esperado), "Los autovalores no coinciden con los analíticos"
assert np.max(residuos) < 1e-12, "Los autopares no satisfacen A v = λ v"
assert np.allclose(np.linalg.norm(vecs, axis=0), 1.0), "NumPy normaliza los autovectores a norma 1"
print("Obtenido :", vals)
print("Esperado :", esperado)
print("Residuos :", residuos)


### Ejercicio L1.3.2: `eig` frente a `eigh`

Sea $X \in \mathbb{R}^{1000 \times 20}$ aleatoria y $C = X^\top X$, que es simétrica y semidefinida positiva. Vamos a comparar las dos rutinas sobre esta matriz en tres aspectos —resultado, ortogonalidad y costo— y a ver qué pasa cuando `eigh` se usa mal.

**a)** Calculá $C$ y su descomposición espectral con `autopares`, una vez con `eig` y otra con `eigh`.

**b)** Medí el error de ortogonalidad de cada base de autovectores, $\|Q^\top Q - I\|_F$, con `np.linalg.norm`.

**c)** Medí con `timeit` el tiempo de $200$ ejecuciones de `np.linalg.eig(C)` y de `np.linalg.eigh(C)`.

**d)** Aplicá `np.linalg.eigvalsh` a una matriz $M$ de $3 \times 3$ **no simétrica** y a la matriz simétrica `M_sim` que el andamiaje construye reflejando el triángulo inferior de $M$. Compará los dos resultados entre sí y contra `np.linalg.eigvals(M)`.

**Nota:** `timeit.timeit(lambda: f(x), number=200)` devuelve el tiempo total de las $200$ ejecuciones. La norma de Frobenius es la que `np.linalg.norm` calcula por defecto sobre una matriz.

In [ ]:
np.random.seed(0)

# a) Matriz simétrica semidefinida positiva
X = np.random.randn(1000, 20)
C = ...  # COMPLETAR: C = X^T X

val_eig, vec_eig = ...    # COMPLETAR: autopares de C tratándola como matriz general
val_eigh, vec_eigh = ...  # COMPLETAR: autopares de C tratándola como matriz simétrica

# b) Error de ortogonalidad de cada base de autovectores: ||Q^T Q - I||_F
I20 = np.eye(20)
ort_eig = ...   # COMPLETAR
ort_eigh = ...  # COMPLETAR

# c) Costo de cada rutina (200 ejecuciones sobre C)
t_eig = ...   # COMPLETAR: usar timeit.timeit con number=200
t_eigh = ...  # COMPLETAR

# d) eigh aplicada a una matriz NO simétrica
M = np.random.randn(3, 3)
M_sim = np.tril(M) + np.tril(M, -1).T   # la matriz simétrica que eigh "ve" (triángulo inferior)

val_M_eigh = ...     # COMPLETAR: autovalores de M con np.linalg.eigvalsh
val_Msim_eigh = ...  # COMPLETAR: autovalores de M_sim con np.linalg.eigvalsh

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.allclose(np.sort(val_eig.real), np.sort(val_eigh)), "Ambas rutinas deben dar el mismo espectro"
assert np.all(val_eigh > -1e-8), "C = X^T X es semidefinida positiva: no tiene autovalores negativos"
assert ort_eigh < 1e-12, "eigh debe devolver una base ortonormal"
assert np.allclose(val_M_eigh, val_Msim_eigh), "eigh sólo lee un triángulo de la matriz"
assert not np.allclose(np.sort(val_M_eigh), np.sort(np.linalg.eigvals(M).real)), \
    "sobre una matriz no simétrica, eigh no devuelve el espectro de M"

print(f"Ortogonalidad ||Q^T Q - I||_F : eig = {ort_eig:.2e} | eigh = {ort_eigh:.2e}")
print(f"Tiempo de 200 ejecuciones     : eig = {t_eig:.4f} s | eigh = {t_eigh:.4f} s | eig/eigh = {t_eig / t_eigh:.2f}x")
print("Autovalores mínimo y máximo de C :", val_eigh.min(), val_eigh.max())
print("M   con eigvalsh :", np.sort(val_M_eigh))
print("M_sim con eigvalsh:", np.sort(val_Msim_eigh))
print("M   con eigvals  :", np.sort(np.linalg.eigvals(M).real))


## Parte 2: Métodos Iterativos

Las rutinas de la Parte 1 calculan el espectro **completo** con un costo de $O(n^3)$. Cuando $n$ es grande y sólo se necesitan unas pocas direcciones, hay una alternativa mucho más barata que además no requiere modificar la matriz: aplicarla repetidamente a un vector.

**El método de la potencia.** Supongamos que $A$ es diagonalizable, con autovalores ordenados por módulo

$$|\lambda_1| > |\lambda_2| \ge \dots \ge |\lambda_n|,$$

y autovectores $v_1, \dots, v_n$. Un vector inicial cualquiera se escribe en esa base como $x_0 = \sum_{i=1}^n c_i v_i$, y aplicando $A$ repetidamente,

$$A^k x_0 = \sum_{i=1}^{n} c_i \lambda_i^k v_i = \lambda_1^k \left[ c_1 v_1 + \sum_{i=2}^{n} c_i \left(\frac{\lambda_i}{\lambda_1}\right)^{k} v_i \right].$$

Como $|\lambda_i / \lambda_1| < 1$ para $i \ge 2$, todos los términos de la suma se desvanecen y la **dirección** de $A^k x_0$ converge a la de $v_1$ (siempre que $c_1 \neq 0$, algo que un vector inicial aleatorio cumple con probabilidad $1$). El error decae como

$$\left|\frac{\lambda_2}{\lambda_1}\right|^{k},$$

de modo que la convergencia es rápida cuando el autovalor dominante está bien separado del siguiente, y lenta cuando $|\lambda_2| \approx |\lambda_1|$.

El factor $\lambda_1^k$ crece o decae exponencialmente y desborda la aritmética de punto flotante, así que en cada paso se normaliza:

$$x_{k+1} = \frac{A x_k}{\|A x_k\|}.$$

Una vez estimada la dirección, el autovalor se recupera con el **cociente de Rayleigh**

$$\rho(A, x) = \frac{x^\top A x}{x^\top x},$$

que devuelve exactamente $\lambda$ cuando $x$ es un autovector. Cada iteración cuesta un producto matriz-vector, $O(n^2)$, y el criterio de parada natural es que el residuo $\|A x_k - \rho(A,x_k)\,x_k\|$ baje de una tolerancia.

**Cuándo falla.** La demostración pide dos cosas, y cada una da un modo de falla distinto:

- Si hay **empate en módulo**, $|\lambda_1| = |\lambda_2|$ con $\lambda_1 \neq \lambda_2$ (por ejemplo $\lambda = \pm a$, o un par complejo conjugado), el cociente $(\lambda_2/\lambda_1)^k$ no tiende a cero y la iteración oscila indefinidamente.
- Si el autovalor dominante está **repetido** ($\lambda_1 = \lambda_2$), el método converge, pero a un autovector cualquiera del autoespacio: cuál depende del $x_0$ elegido.
- Si la matriz **no es diagonalizable**, la expansión en la base de autovectores no existe. El método todavía converge al único autovector disponible, pero a velocidad $O(1/k)$ en lugar de geométrica.

**Potencia inversa y desplazamiento.** Los autovalores de $(A - \sigma I)^{-1}$ son $\dfrac{1}{\lambda_i - \sigma}$, y el mayor de ellos en módulo corresponde al $\lambda_i$ **más cercano a $\sigma$**. Aplicar el método de la potencia a esa matriz permite entonces buscar un autovalor arbitrario, no sólo el dominante:

$$x_{k+1} = \frac{(A - \sigma I)^{-1} x_k}{\|(A - \sigma I)^{-1} x_k\|}.$$

Con $\sigma = 0$ se obtiene la **potencia inversa** clásica, que converge al autovalor de menor módulo. En la implementación nunca se calcula la inversa: se resuelve el sistema $(A - \sigma I)\,y = x_k$ en cada paso, lo que es más barato y más estable. Cuando $\sigma$ está muy cerca de un autovalor la matriz $A - \sigma I$ queda mal condicionada, pero el error que eso introduce cae justamente en la dirección del autovector buscado, así que el método funciona igual: es una de las inestabilidades benignas del análisis numérico.

### Ejercicio L1.3.3: Método de la Potencia

**a)** Implementá `coeficiente_rayleigh(A, x)`, que calcula $\rho(A,x) = \dfrac{x^\top A x}{x^\top x}$.

**b)** Implementá `metodo_potencia(A, x0=None, tol=1e-10, max_iter=1000)` siguiendo el andamiaje: normalizar el vector inicial, y en cada iteración aplicar $A$, volver a normalizar y estimar el autovalor con el cociente de Rayleigh. La función corta apenas `residuo_autopar` baja de `tol` y devuelve la tupla `(v, lam, k)` con el autovector unitario, el autovalor y la cantidad de iteraciones realizadas.

Aplicala a $M = \begin{bmatrix} 4 & 2 \\ 1 & 3 \end{bmatrix}$, cuyos autovalores son $5$ y $2$.

**Nota:** el producto matriz-vector en NumPy es `A @ x` y la norma euclídea es `np.linalg.norm(x)`. El bloque de verificación compara contra `autopares` y controla que el autovector obtenido sea colineal con el de NumPy (pueden diferir en el signo).

In [ ]:
def coeficiente_rayleigh(A, x):
    """
    Calcula el cociente de Rayleigh (x^T A x) / (x^T x).

    Args:
        A: matriz cuadrada (n x n)
        x: vector no nulo (array 1D de largo n)

    Returns:
        rho: estimación del autovalor asociado a la dirección de x (escalar)
    """
    return ...  # COMPLETAR


def metodo_potencia(A, x0=None, tol=1e-10, max_iter=1000):
    """
    Aproxima el autovalor dominante de A y su autovector asociado.

    Args:
        A: matriz cuadrada (n x n)
        x0: vector inicial (array 1D de largo n); si es None se toma uno aleatorio
        tol: tolerancia sobre el residuo ||A v - lam v|| / ||v||
        max_iter: cantidad máxima de iteraciones

    Returns:
        v: autovector dominante, de norma 1 (array 1D de largo n)
        lam: autovalor dominante (escalar)
        k: iteraciones realizadas
    """
    n = A.shape[0]
    x = np.random.rand(n) if x0 is None else np.asarray(x0, dtype=float).copy()
    x = x / np.linalg.norm(x)
    lam = coeficiente_rayleigh(A, x)

    for k in range(1, max_iter + 1):
        x = ...    # COMPLETAR: aplicar A al vector actual
        x = ...    # COMPLETAR: normalizar
        lam = ...  # COMPLETAR: estimar el autovalor con el cociente de Rayleigh
        if residuo_autopar(A, lam, x) < tol:
            return x, lam, k

    return x, lam, max_iter


# ── Verificación ─────────────────────────────────────────────────────────────
M = np.array([[4.0, 2.0],
              [1.0, 3.0]])

v_pot, lam_pot, k_pot = metodo_potencia(M, x0=np.array([1.0, 0.0]))
vals_M, vecs_M = autopares(M)
cos_v = np.dot(v_pot, vecs_M[:, 0]) / np.linalg.norm(vecs_M[:, 0])

assert np.isclose(lam_pot, 5.0), "El autovalor dominante de M es 5"
assert residuo_autopar(M, lam_pot, v_pot) < 1e-8, "El autopar no satisface A v = λ v"
assert np.isclose(abs(cos_v), 1.0), "El autovector debe ser colineal con el que devuelve NumPy"
print(f"Potencia : λ = {lam_pot:.12f} | v = {v_pot} | iteraciones = {k_pot}")
print(f"NumPy    : λ = {vals_M[0]:.12f} | v = {vecs_M[:, 0]}")


### Ejercicio L1.3.4: Velocidad de Convergencia

La teoría dice que el error del método de la potencia decae como $|\lambda_2/\lambda_1|^k$. Vamos a comprobarlo sobre una matriz con espectro elegido de antemano: si $Q$ es ortogonal y $\Lambda = \operatorname{diag}(8, 4, 2, 1)$, entonces $A = Q\Lambda Q^\top$ es simétrica y tiene exactamente esos autovalores, de modo que $|\lambda_2/\lambda_1| = 1/2$ es conocido.

**a)** Implementá `historial_residuos(A, x0, num_iter)`, que corre `num_iter` iteraciones del método de la potencia **sin criterio de parada** y devuelve el vector con el residuo de cada iteración.

**b)** Corré $30$ iteraciones sobre $A$ y guardá en `razon` el cociente teórico $|\lambda_2/\lambda_1|$. El andamiaje grafica el residuo observado en escala logarítmica contra la cota teórica $C\,|\lambda_2/\lambda_1|^k$ y calcula los factores de decaimiento $r_{k+1}/r_k$.

**c)** OPCIONAL: repetí el experimento con el espectro $\Lambda = \operatorname{diag}(8, 7.9, 2, 1)$ y observá cuántas iteraciones hacen falta ahora para el mismo residuo.

**Nota:** una matriz ortogonal aleatoria se obtiene factorizando una matriz aleatoria con `np.linalg.qr`. En escala logarítmica, un decaimiento geométrico se ve como una recta: la pendiente es $\log|\lambda_2/\lambda_1|$.

In [ ]:
np.random.seed(1)


def historial_residuos(A, x0, num_iter):
    """
    Corre num_iter iteraciones del método de la potencia, sin criterio de parada,
    y registra el residuo del autopar en cada una.

    Args:
        A: matriz cuadrada (n x n)
        x0: vector inicial (array 1D de largo n)
        num_iter: cantidad de iteraciones a realizar

    Returns:
        residuos: residuo de cada iteración (array de largo num_iter)
    """
    x = np.asarray(x0, dtype=float).copy()
    x = x / np.linalg.norm(x)
    residuos = np.zeros(num_iter)

    for k in range(num_iter):
        x = ...            # COMPLETAR: aplicar A al vector actual
        x = ...            # COMPLETAR: normalizar
        residuos[k] = ...  # COMPLETAR: residuo del autopar (cociente de Rayleigh, x)

    return residuos


# b) Matriz simétrica con espectro conocido: λ = 8, 4, 2, 1
Q_rand, _ = np.linalg.qr(np.random.randn(4, 4))
espectro = np.array([8.0, 4.0, 2.0, 1.0])
A_conv = Q_rand @ np.diag(espectro) @ Q_rand.T

num_iter = 30
res = historial_residuos(A_conv, np.ones(4), num_iter)
razon = ...  # COMPLETAR: cociente teórico |λ2 / λ1| a partir de espectro

factores = res[1:] / res[:-1]

# ── Verificación ─────────────────────────────────────────────────────────────
ks = np.arange(1, num_iter + 1)
plt.figure(figsize=(7, 4))
plt.semilogy(ks, res, "o-", label="residuo observado")
plt.semilogy(ks, res[0] * razon ** (ks - 1), "--", label=r"cota $C\,|\lambda_2/\lambda_1|^k$")
plt.xlabel("iteración $k$")
plt.ylabel("residuo")
plt.title("Convergencia del método de la potencia")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

assert np.isclose(razon, 0.5), "El cociente teórico para este espectro es 4/8 = 0.5"
assert np.isclose(np.mean(factores[-8:]), razon, atol=0.05), \
    "El residuo debe decaer asintóticamente por un factor |λ2/λ1| en cada iteración"
print("Factor de decaimiento observado (promedio últimas 8 iteraciones):", np.mean(factores[-8:]))
print("Factor teórico |λ2/λ1|                                         :", razon)
print("Residuo final                                                  :", res[-1])


### Ejercicio L1.3.5: Casos de Falla y Costo Computacional

**a)** Calculá analíticamente los autovalores y autovectores de las tres matrices siguientes y aplicale a cada una `metodo_potencia` con el vector inicial que indica el andamiaje. Explicá en cada caso qué se observa y con cuál de las hipótesis de la demostración choca:

$$A_1 = \begin{bmatrix} 1 & 0 \\ 0 & -1\end{bmatrix}, \qquad
  A_2 = \begin{bmatrix} 2 & 0 \\ 0 & 2\end{bmatrix}, \qquad
  A_3 = \begin{bmatrix} 2 & 1 \\ 0 & 2\end{bmatrix}.$$

Para $A_2$, corré el método desde **dos** vectores iniciales distintos y compará los autovectores obtenidos.

**b)** Compará el costo del método de la potencia contra la descomposición completa sobre una matriz simétrica de $600 \times 600$ con un autovalor dominante bien separado (el andamiaje la construye sumándole a una matriz simétrica aleatoria un término de rango uno). Medí con `timeit` el tiempo del método de la potencia y el de `np.linalg.eigh`.

**Nota:** el método de la potencia devuelve `k = max_iter` cuando nunca alcanzó la tolerancia; ése es el indicador de que no convergió. Recordá que cada iteración cuesta $O(n^2)$ frente a los $O(n^3)$ de la descomposición completa, y que sólo entrega **un** autopar.

In [ ]:
np.random.seed(2)

# a) Tres casos de falla
A1 = np.array([[1.0, 0.0], [0.0, -1.0]])    # |λ1| = |λ2|, autovalores distintos
A2 = np.array([[2.0, 0.0], [0.0, 2.0]])     # autovalor repetido
A3 = np.array([[2.0, 1.0], [0.0, 2.0]])     # no diagonalizable (bloque de Jordan)

x0 = np.array([1.0, 1.0])

v1, lam1, k1 = ...  # COMPLETAR: metodo_potencia sobre A1 desde x0
v2a, lam2a, k2a = ...  # COMPLETAR: metodo_potencia sobre A2 desde x0
v2b, lam2b, k2b = ...  # COMPLETAR: metodo_potencia sobre A2 desde [1, 3]
v3, lam3, k3 = ...  # COMPLETAR: metodo_potencia sobre A3 desde x0

# b) Costo: matriz simétrica grande con autovalor dominante separado
n_g = 600
W = np.random.randn(n_g, n_g)
W = (W + W.T) / (2.0 * np.sqrt(n_g))          # espectro acotado, sin dirección privilegiada
u = np.random.randn(n_g)
u = u / np.linalg.norm(u)
S_grande = W + 6.0 * np.outer(u, u)           # se "planta" un autovalor dominante ≈ 6

v_g, lam_g, k_g = ...  # COMPLETAR: metodo_potencia sobre S_grande desde np.ones(n_g), tol=1e-8
t_potencia = ...       # COMPLETAR: timeit de 5 ejecuciones del método de la potencia
t_eigh = ...           # COMPLETAR: timeit de 5 ejecuciones de np.linalg.eigh(S_grande)

# ── Verificación ─────────────────────────────────────────────────────────────
assert k1 == 1000 and residuo_autopar(A1, lam1, v1) > 1e-6, \
    "A1 tiene dos autovalores de igual módulo: el método no debe converger"
assert np.isclose(lam2a, 2.0) and np.isclose(lam2b, 2.0), "El autovalor de A2 es 2 (repetido)"
assert not np.isclose(abs(np.dot(v2a, v2b)), 1.0), \
    "Con autovalor repetido, el autovector obtenido depende del vector inicial"
assert k3 == 1000 and residuo_autopar(A3, lam3, v3) > 1e-6, \
    "A3 no es diagonalizable: la convergencia es O(1/k), no geométrica"
assert np.isclose(lam_g, np.linalg.eigvalsh(S_grande)[-1], atol=1e-6), \
    "El método debe encontrar el autovalor dominante de S_grande"
assert t_potencia < t_eigh, "Un solo autopar por iteraciones O(n^2) debe salir más barato que O(n^3)"

print(f"A1: λ = {lam1:.4f} | v = {v1} | iteraciones = {k1} | residuo = {residuo_autopar(A1, lam1, v1):.2e}")
print(f"A2: λ = {lam2a:.4f} desde {x0} → v = {v2a}")
print(f"A2: λ = {lam2b:.4f} desde [1, 3] → v = {v2b}")
print(f"A3: λ = {lam3:.6f} | v = {v3} | iteraciones = {k3} | residuo = {residuo_autopar(A3, lam3, v3):.2e}")
print(f"\nS_grande ({n_g}x{n_g}): λ_dominante = {lam_g:.6f} en {k_g} iteraciones")
print(f"  potencia (5 ejec.) : {t_potencia:.4f} s")
print(f"  eigh     (5 ejec.) : {t_eigh:.4f} s   → cociente {t_eigh / t_potencia:.1f}x")


### Ejercicio L1.3.6: Potencia Inversa con Desplazamiento

Implementá `potencia_inversa(A, sigma=0.0, x0=None, tol=1e-10, max_iter=1000)`, que aplica el método de la potencia a $(A - \sigma I)^{-1}$ y devuelve el autopar de $A$ cuyo autovalor es **más cercano a $\sigma$**, con la misma firma de salida `(v, lam, k)` que `metodo_potencia`.

**a)** Completá la iteración: en cada paso hay que resolver el sistema $(A - \sigma I)\,y = x_k$, normalizar la solución y estimar el autovalor con el cociente de Rayleigh **sobre $A$**, no sobre $A - \sigma I$.

**b)** Aplicala a $A = \begin{bmatrix} 1 & 2 \\ 3 & 1\end{bmatrix}$ con $\sigma = 0$ (autovalor de menor módulo, $1 - \sqrt{6}$) y con $\sigma = 2$ (el más cercano a $2$ es $1 + \sqrt{6}$), y a la matriz `A_conv` del Ejercicio L1.3.4 con $\sigma = 1.9$, que debe devolver el autovalor $2$.

**Nota:** usá `np.linalg.solve(B, x)`, nunca `np.linalg.inv`. Resolver el sistema cuesta $O(n^3)$ la primera vez, pero como la matriz $B = A - \sigma I$ no cambia entre iteraciones, en una implementación seria se factoriza una sola vez (por ejemplo con `scipy.linalg.lu_factor` y `lu_solve`) y cada iteración vuelve a costar $O(n^2)$.

In [ ]:
def potencia_inversa(A, sigma=0.0, x0=None, tol=1e-10, max_iter=1000):
    """
    Aproxima el autovalor de A más cercano a sigma y su autovector asociado,
    aplicando el método de la potencia a (A - sigma I)^(-1).

    Args:
        A: matriz cuadrada (n x n)
        sigma: desplazamiento; con sigma = 0 se obtiene el autovalor de menor módulo
        x0: vector inicial (array 1D de largo n); si es None se toma uno aleatorio
        tol: tolerancia sobre el residuo ||A v - lam v|| / ||v||
        max_iter: cantidad máxima de iteraciones

    Returns:
        v: autovector de norma 1 (array 1D de largo n)
        lam: autovalor de A más cercano a sigma (escalar)
        k: iteraciones realizadas
    """
    n = A.shape[0]
    B = A - sigma * np.eye(n)
    x = np.random.rand(n) if x0 is None else np.asarray(x0, dtype=float).copy()
    x = x / np.linalg.norm(x)
    lam = coeficiente_rayleigh(A, x)

    for k in range(1, max_iter + 1):
        y = ...    # COMPLETAR: resolver B y = x
        x = ...    # COMPLETAR: normalizar y
        lam = ...  # COMPLETAR: cociente de Rayleigh de A (no de B) en la dirección x
        if residuo_autopar(A, lam, x) < tol:
            return x, lam, k

    return x, lam, max_iter


# b) Aplicación
v_min, lam_min, k_min = ...      # COMPLETAR: A con sigma = 0
v_cerca, lam_cerca, k_cerca = ...  # COMPLETAR: A con sigma = 2
v_conv, lam_conv, k_conv = ...   # COMPLETAR: A_conv con sigma = 1.9

# ── Verificación ─────────────────────────────────────────────────────────────
assert np.isclose(lam_min, 1 - np.sqrt(6)), "Con σ = 0 se obtiene el autovalor de menor módulo"
assert np.isclose(lam_cerca, 1 + np.sqrt(6)), "Con σ = 2 el autovalor más cercano es 1 + √6"
assert np.isclose(lam_conv, 2.0, atol=1e-8), "El autovalor de A_conv más cercano a 1.9 es 2"
assert residuo_autopar(A, lam_min, v_min) < 1e-8, "El autopar debe satisfacer A v = λ v"
assert residuo_autopar(A_conv, lam_conv, v_conv) < 1e-8, "El autopar debe satisfacer A v = λ v"
print(f"σ = 0.0 → λ = {lam_min:.12f} (esperado {1 - np.sqrt(6):.12f}) en {k_min} iteraciones")
print(f"σ = 2.0 → λ = {lam_cerca:.12f} (esperado {1 + np.sqrt(6):.12f}) en {k_cerca} iteraciones")
print(f"A_conv, σ = 1.9 → λ = {lam_conv:.12f} (esperado 2.0) en {k_conv} iteraciones")


## Parte 3: De los Autovectores a las Direcciones Principales

El método de la potencia entrega **un** autopar. Para obtener varios se usa la **deflación de Hotelling**, que aprovecha el teorema espectral. Si $A$ es simétrica con base ortonormal de autovectores y $(\lambda_1, v_1)$ es su autopar dominante, la matriz

$$A' = A - \lambda_1 v_1 v_1^\top$$

tiene exactamente los mismos autovectores que $A$, con los mismos autovalores salvo el primero, que pasa a valer $0$: para $j \neq 1$, $A' v_j = A v_j - \lambda_1 v_1 (v_1^\top v_j) = \lambda_j v_j$, porque $v_1^\top v_j = 0$; y $A' v_1 = \lambda_1 v_1 - \lambda_1 v_1 = 0$. Aplicando el método de la potencia a $A'$ se obtiene $(\lambda_2, v_2)$, y repitiendo el procedimiento $k$ veces se obtienen los $k$ autopares dominantes. La ortogonalidad de los autovectores es esencial, así que la deflación en esta forma sirve para matrices simétricas.

**La conexión con PCA.** Sea $X \in \mathbb{R}^{d \times n}$ una matriz de datos cuyas **columnas** son las $n$ observaciones, y sea $X_c$ la matriz centrada (a cada fila se le resta su media). La matriz de covarianza empírica es

$$S = \frac{1}{n} X_c X_c^\top \in \mathbb{R}^{d \times d},$$

simétrica y semidefinida positiva. La varianza de los datos proyectados sobre una dirección unitaria $w$ es

$$\operatorname{Var}(w^\top X) = \frac{1}{n}\sum_{i=1}^n \left(w^\top x_i^{(c)}\right)^2 = w^\top S\, w = \rho(S, w),$$

es decir, exactamente el cociente de Rayleigh de la Parte 2. Buscar la dirección de **máxima varianza** es maximizar el cociente de Rayleigh, y el teorema espectral dice que el máximo se alcanza en el autovector dominante de $S$, con valor $\lambda_1$. La segunda dirección principal es el autovector dominante entre los ortogonales al primero, que es justo lo que devuelve la deflación. La fracción de varianza que explica cada componente es

$$\frac{\lambda_i}{\operatorname{tr}(S)} = \frac{\lambda_i}{\sum_{j=1}^d \lambda_j},$$

ya que la traza de $S$ es la varianza total de los datos. Con esto queda armado el núcleo de PCA, que el Laboratorio 1.4 retoma para comprimir y reconstruir imágenes.

### Ejercicio L1.3.7: Los $k$ Autovectores Dominantes por Deflación

Implementá `k_autovectores_dominantes(A, k, tol=1e-10, max_iter=2000)` para `A` simétrica: en cada uno de los $k$ pasos, hallá el autopar dominante de la matriz actual con `metodo_potencia`, guardalo, y deflacioná restando $\lambda v v^\top$. La función devuelve la tupla `(V, lams)`, donde `V` es la matriz $k \times n$ cuyas **filas** son los autovectores y `lams` el vector de autovalores.

Aplicala con $k = 3$ a la matriz `A_conv` del Ejercicio L1.3.4, cuyo espectro es $\{8, 4, 2, 1\}$.

**Nota:** el producto exterior $v v^\top$ se calcula con `np.outer(v, v)`. Trabajá sobre una **copia** de `A` (`A.astype(float).copy()`), porque la deflación modifica la matriz y el bloque de verificación necesita la original para calcular los residuos.

In [ ]:
def k_autovectores_dominantes(A, k, tol=1e-10, max_iter=2000):
    """
    Calcula los k autopares dominantes de una matriz simétrica por deflación.

    Args:
        A: matriz simétrica (n x n)
        k: cantidad de autopares a calcular
        tol: tolerancia del método de la potencia
        max_iter: iteraciones máximas de cada llamada al método de la potencia

    Returns:
        V: matriz cuyas FILAS son los k autovectores, ordenados por |λ| decreciente (k x n)
        lams: autovalores correspondientes (array de largo k)
    """
    A_def = A.astype(float).copy()
    vectores, valores = [], []

    for _ in range(k):
        v, lam, _ = ...  # COMPLETAR: autopar dominante de la matriz actual
        vectores.append(v)
        valores.append(lam)
        A_def = ...      # COMPLETAR: deflación de Hotelling

    return np.array(vectores), np.array(valores)


V3, lams3 = ...  # COMPLETAR: los 3 autopares dominantes de A_conv

# ── Verificación ─────────────────────────────────────────────────────────────
residuos3 = np.array([residuo_autopar(A_conv, lams3[i], V3[i]) for i in range(3)])
ort3 = np.max(np.abs(V3 @ V3.T - np.eye(3)))

assert np.allclose(lams3, espectro[:3], atol=1e-6), "Los autovalores deben ser 8, 4 y 2"
assert np.max(residuos3) < 1e-6, "Cada autopar debe satisfacer A v = λ v sobre la matriz ORIGINAL"
assert ort3 < 1e-6, "Los autovectores de una matriz simétrica son ortonormales"
print("Obtenido :", lams3)
print("Esperado :", espectro[:3])
print("Residuos sobre A_conv          :", residuos3)
print("Error de ortogonalidad de V3   :", ort3)


### Ejercicio L1.3.8: Direcciones Principales de un Conjunto de Datos

Implementá `direcciones_principales(X, k)`, que recibe una matriz de datos $X \in \mathbb{R}^{d \times n}$ (columnas = observaciones) y devuelve la tupla `(W, var_explicada, X_proy)`:

- `W`: matriz $k \times d$ cuyas filas son las $k$ direcciones principales, obtenidas con `k_autovectores_dominantes` sobre la matriz de covarianza.
- `var_explicada`: fracción de varianza explicada por cada componente, $\lambda_i / \operatorname{tr}(S)$.
- `X_proy`: los datos centrados proyectados sobre esas direcciones, $W X_c$, de tamaño $k \times n$.

El andamiaje genera una nube gaussiana de $2000$ puntos en $\mathbb{R}^2$ con desvíos $3$ y $1$ a lo largo de dos ejes ortogonales, rotada $30°$ y desplazada del origen. Las direcciones principales tienen que recuperar los ejes de la rotación, y la varianza explicada debe repartirse aproximadamente como $9/10$ y $1/10$.

**Nota:** para centrar usá `X.mean(axis=1, keepdims=True)`, que preserva la forma de columna y permite restar por *broadcasting*. La traza se calcula con `np.trace`. El bloque de verificación compara contra `sklearn.decomposition.PCA`, que usa el estimador insesgado con $n-1$ en el denominador; como la fracción de varianza explicada es un cociente, ese factor se cancela y ambos resultados coinciden.

In [ ]:
np.random.seed(4)


def direcciones_principales(X, k):
    """
    Calcula las k direcciones principales de un conjunto de datos.

    Args:
        X: matriz de datos (d x n), con las observaciones como columnas
        k: cantidad de componentes principales

    Returns:
        W: direcciones principales como filas (k x d)
        var_explicada: fracción de varianza explicada por cada componente (array de largo k)
        X_proy: datos centrados proyectados sobre las k direcciones (k x n)
    """
    n = X.shape[1]
    X_c = ...  # COMPLETAR: centrar los datos (restar la media de cada fila)
    S = ...    # COMPLETAR: matriz de covarianza (1/n) X_c X_c^T

    W, lams = ...          # COMPLETAR: k autopares dominantes de S
    var_explicada = ...    # COMPLETAR: λ_i / traza(S)
    X_proy = ...           # COMPLETAR: proyección de los datos centrados

    return W, var_explicada, X_proy


# Datos sintéticos: nube gaussiana con desvíos 3 y 1, rotada 30° y desplazada
n_datos = 2000
theta = np.deg2rad(30.0)
R = np.array([[np.cos(theta), -np.sin(theta)],
              [np.sin(theta),  np.cos(theta)]])
X_datos = R @ (np.diag([3.0, 1.0]) @ np.random.randn(2, n_datos)) + np.array([[5.0], [-2.0]])

W, var_exp, X_proy = ...  # COMPLETAR: 2 direcciones principales de X_datos

# ── Verificación contra scikit-learn ─────────────────────────────────────────
from sklearn.decomposition import PCA

pca_ref = PCA(n_components=2).fit(X_datos.T)
cos_dir = np.abs(np.sum(W * pca_ref.components_, axis=1))

media = X_datos.mean(axis=1)
escala = 2.5 * np.sqrt(var_exp * np.trace(np.cov(X_datos, bias=True)))
plt.figure(figsize=(6, 6))
plt.scatter(X_datos[0], X_datos[1], s=6, alpha=0.25, label="datos")
for i, color in enumerate(["crimson", "darkorange"]):
    plt.arrow(media[0], media[1], escala[i] * W[i, 0], escala[i] * W[i, 1],
              width=0.05, color=color, length_includes_head=True,
              label=f"CP {i + 1} ({var_exp[i] * 100:.1f}% var.)")
plt.axis("equal")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.title("Direcciones principales de la nube de datos")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

assert np.allclose(var_exp, pca_ref.explained_variance_ratio_, atol=1e-6), \
    "La varianza explicada debe coincidir con la de scikit-learn"
assert np.allclose(cos_dir, 1.0, atol=1e-5), \
    "Cada dirección debe ser colineal con la componente correspondiente de scikit-learn"
assert np.allclose(np.abs(W @ R), np.eye(2), atol=0.05), \
    "Las direcciones principales deben recuperar los ejes de la rotación"
assert np.allclose(X_proy.mean(axis=1), 0.0, atol=1e-10), "Los datos proyectados quedan centrados"
print("Varianza explicada (propia)   :", np.round(var_exp, 4))
print("Varianza explicada (sklearn)  :", np.round(pca_ref.explained_variance_ratio_, 4))
print("Varianza explicada (teórica)  :", np.round(np.array([9.0, 1.0]) / 10.0, 4))
print("|cos| con las componentes de sklearn:", np.round(cos_dir, 8))


## Conclusiones

En este laboratorio hemos explorado:

1. **Descomposición espectral con NumPy**: `eig` sirve para matrices generales y devuelve autovalores complejos en orden arbitrario; `eigh` explota la simetría para trabajar a menor costo, con autovalores reales ordenados y una base ortonormal garantizada. La contracara es que `eigh` lee un solo triángulo de la matriz, así que aplicada a una matriz no simétrica devuelve, sin avisar, el espectro de otra matriz: verificar con el residuo $\|Av - \lambda v\|$ no es opcional.
2. **Método de la potencia**: multiplicar repetidamente por $A$ y normalizar hace que la dirección dominante se imponga sobre las demás, y el cociente de Rayleigh recupera el autovalor. El error decae como $|\lambda_2/\lambda_1|^k$, lo que se verifica como una recta en escala logarítmica: la separación entre los dos primeros autovalores, y no el tamaño de la matriz, es lo que determina cuántas iteraciones hacen falta.
3. **Casos de falla**: autovalores de igual módulo y signo opuesto producen oscilación permanente; un autovalor repetido da convergencia a un autovector cualquiera del autoespacio, dependiente del vector inicial; y una matriz no diagonalizable degrada la convergencia de geométrica a $O(1/k)$. Cada falla corresponde a una hipótesis de la demostración.
4. **Potencia inversa y desplazamiento**: como los autovalores de $(A - \sigma I)^{-1}$ son $1/(\lambda_i - \sigma)$, desplazar y resolver un sistema en cada iteración permite apuntar a cualquier parte del espectro, no sólo al extremo dominante. El costo por iteración vuelve a ser $O(n^2)$ si la factorización de $A - \sigma I$ se calcula una sola vez.
5. **Deflación y direcciones principales**: restar $\lambda v v^\top$ anula el autovalor ya encontrado y deja intactos los demás, lo que permite obtener los $k$ autovectores dominantes de una matriz simétrica encadenando llamadas al método de la potencia. Aplicado a la matriz de covarianza, ese procedimiento es exactamente PCA: maximizar la varianza de la proyección es maximizar el cociente de Rayleigh, y la varianza explicada por cada componente es $\lambda_i/\operatorname{tr}(S)$. Sobre esta base se construye el Laboratorio 1.4.

## Declaración de uso de inteligencia artificial

> **Política del curso (syllabus).** Se permite usar herramientas de IA (ChatGPT, Copilot, Claude, etc.)
> *como apoyo para el aprendizaje*: entender conceptos, explorar ideas, depurar código o buscar
> explicaciones alternativas. **No** se permite usarlas para **resolver los ejercicios evaluados** ni
> para **verificar las respuestas antes de entregar**. Se espera que cada estudiante resuelva todos los
> problemas por sí mismo/a.

Completá esta declaración **escribiendo tu respuesta** donde aparece «…» (doble clic en esta celda para
editarla y luego Ctrl/Cmd + Enter para volver a renderizarla):

**1. ¿Usaste herramientas de IA en este laboratorio?** (Sí / No): «…»

**2. ¿Cuál(es)?** (ChatGPT, Copilot, Claude, …; escribí "ninguna" si no usaste): «…»

**3. ¿Para qué la(s) usaste?** Usos permitidos: entender conceptos, explorar ideas, depurar código,
buscar explicaciones alternativas. Escribí los que apliquen: «…»

**4. Detalle breve:** «En qué ejercicios y de qué manera. Ej.: "Usé Claude para entender el cociente de
Rayleigh antes del ejercicio L1.3.3."»

**Declaración de honestidad académica.** Declaro que resolví los ejercicios de este laboratorio por mí
mismo/a y que no utilicé herramientas de IA para resolver los ejercicios evaluados ni para verificar mis
respuestas antes de entregar, de acuerdo con la política del curso.

**Nombre y apellido:** «…»          **Fecha:** «…»